# M02.01 — Structured Output

An LLM returns text. To use it as a software component, other code must be able to consume that text reliably. In this notebook we extract the fields of a boarding pass (an image) and make the output progressively more controlled:

1. Free-text extraction
2. Asking for JSON in the prompt
3. JSON mode
4. A JSON schema, enforced by the API
5. A Pydantic model, validated at runtime
6. Cost

The boarding pass covers every basic type a schema can ask for: strings, an integer, a boolean, a date and an enum. In the exercises you apply the same steps to the medical invoices, and add one more type: `tags`, a list of values from a fixed set (an array of enum values). The `MedicalInvoice` object you end up with is the input of every later module.

## Why JSON?

- **Interoperability**: every language can parse it.
- **Hierarchical**: nested objects and lists map onto real data (an invoice with several treatments, a list of citations).
- **Validatable**: a schema says exactly which fields, of which type, must be present.

## Setup

In [ ]:
import json
import re
from datetime import date
from enum import Enum
from pathlib import Path

from IPython.display import Image, display
from pydantic import BaseModel, Field, ValidationError

from agentic_workshop.client import get_client
from agentic_workshop.config import CHAT_MODEL, DATA_DIR, INVOICES_DIR, OLD_MODEL
from agentic_workshop.costs import PRICES, print_cost, usage_cost
from agentic_workshop.invoices import image_to_data_url

client = get_client()
MODEL = CHAT_MODEL

## The boarding pass - Example

`data/boarding_pass/boarding_pass.png` is an example boarding pass for a flight from Geneva to Zurich. The model reads images directly: we send the PNG as a base64 data URL (`image_to_data_url`) next to our text prompt.

In [ ]:
BOARDING_PASS = DATA_DIR / "boarding_pass" / "boarding_pass.png"
display(Image(filename=BOARDING_PASS, width=800))


def image_input(prompt: str, path: Path = BOARDING_PASS) -> list[dict]:
    """A user message with our prompt and an image."""
    return [{"role": "user", "content": [
        {"type": "input_text", "text": prompt},
        {"type": "input_image", "image_url": image_to_data_url(path)},
    ]}]

## 1. Default extraction

We ask for the fields, with no indication of the format.

In [ ]:
PROMPT = """You are a document processing assistant. Extract the following details from this boarding pass:
- Passenger name
- Origin city
- Destination city
- Departure date
- Seat row and seat letter
- Whether a cabin bag is allowed
- Travel class"""

response = client.responses.create(model=MODEL, input=image_input(PROMPT))
print(response.output_text)

The model extracted the fields, but as free text whose layout can change from one call to the next. Code that consumes it would have to attempt to parse a non-standard input each time.

## 2. Can we get JSON just by asking?

Here we use `OLD_MODEL` which points to `gpt-4o-mini` rather than `gpt-6-luna` which is used for the rest of the exercise. The older model is generally less reliable on certain tasks like structured extraction so while it happens *much* less often now it's still useful to be familiar with this kind of edge-case.

In [ ]:
response = client.responses.create(model=OLD_MODEL, input=image_input(PROMPT + "\n\nGive me the result in JSON format."))
print(response.output_text)

In [ ]:
# Most likely this will fail
json.loads(response.output_text)

A model asked for JSON in the prompt often wraps it in a Markdown code block (`` ```json ... ``` ``), which `json.loads` rejects. Nothing guarantees the format: we have to clean the text ourselves.

In [ ]:
# Now we should see it load it properly!
json_str = re.sub(r"^```(?:json)?\s*", "", response.output_text.strip())
json_str = re.sub(r"\s*```$", "", json_str)
json.loads(json_str)

And here we switch back to using `MODEL`, so this time it should work without an issue.

In [ ]:
response = client.responses.create(model=MODEL, input=image_input(PROMPT + "\n\nGive me the result in JSON format."))
print(response.output_text)

In [ ]:
# Newer model handles this without problems
json.loads(response.output_text)

## 3. JSON mode

The API can force the output to be valid JSON: `text={"format": {"type": "json_object"}}`. (The prompt must still mention JSON.)

In [ ]:
response = client.responses.create(
    model=MODEL,
    input=image_input(PROMPT + "\n\nGive me the result in JSON format."),
    text={"format": {"type": "json_object"}},
)
json.loads(response.output_text)

The output now always parses. But its *shape* is still up to the model: the keys can be `passenger_name`, `Passenger name` or `passengerName`, the date `14NOV26` or `2026-11-14`, the cabin bag `true` or `"1 x 8 kg included"`. Run the cell twice and compare.

## 4. A JSON schema

We pass a JSON schema describing exactly which keys, of which type, we want. With `"strict": True` the API **constrains generation** to the schema. Strict mode requires every property to be listed in `required` and `additionalProperties: false`; an optional value is expressed as a type that also allows `null`.

Each field of the boarding pass shows one type:

| Field | JSON schema | On the pass |
|---|---|---|
| `passenger_name`, `origin`, `destination`, `seat_letter` | `{"type": "string"}` | `MEIER / SOPHIE`, `GENEVA GVA`, the `A` of `23A` |
| `seat_row` | `{"type": "integer"}` | the `23` of `23A` |
| `cabin_bag_allowed` | `{"type": "boolean"}` | `1 x 8 kg INCLUDED` |
| `departure_date` | `{"type": "string", "format": "date"}` | `14NOV26` |
| `travel_class` | `{"type": "string", "enum": ["Economy", "Business"]}` | `ECONOMY` |

JSON has no date type: a date is a string, and `"format": "date"` asks for an ISO-8601 date (`YYYY-MM-DD`). An `enum` restricts a value to a fixed list. A `description` tells the model how to fill a field, such as the name order.

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "passenger_name": {"type": "string", "description": "As: Firstname Middlename Lastname"},
        "origin": {"type": "string", "description": "City of origin."},
        "destination": {"type": "string", "description": "Destination city."},
        "departure_date": {"type": "string", "format": "date", "description": "ISO-8601 formatted date; YYYY-MM-DD."},
        "seat_row": {"type": "integer"},
        "seat_letter": {"type": "string"},
        "cabin_bag_allowed": {"type": "boolean"},
        "travel_class": {"type": "string", "enum": ["Economy", "Business"]},
    },
    "required": ["passenger_name", "origin", "destination", "departure_date", "seat_row", "seat_letter", "cabin_bag_allowed", "travel_class"],
    "additionalProperties": False,
}

response = client.responses.create(
    model=MODEL,
    input=image_input("Extract the boarding pass fields."),
    text={"format": {"type": "json_schema", "name": "BoardingPass", "schema": schema, "strict": True}},
)
json.loads(response.output_text)

The keys and types now always match the schema: `seat_row` is a number, `cabin_bag_allowed` is `true` or `false`, and `travel_class` is exactly `"Economy"` or `"Business"` (never `"ECONOMY"` or `"Eco"`). The *values* can still be wrong: the schema guarantees the shape, not the truth.

## 5. Pydantic

Writing JSON schemas by hand is tedious. A [Pydantic](https://docs.pydantic.dev/latest/) model describes the same thing as a Python class: `client.responses.parse(text_format=...)` turns it into a strict schema, and **validates** the answer back into an object. `output_parsed` is a `BoardingPass`, not a string.

The Python types map onto the JSON types: `str` → `string`, `int` → `integer`, `bool` → `boolean`, `date` → `string` with `"format": "date"`. An enum is a Python `Enum` class. Because `TravelClass` subclasses `str`, a member compares equal to its value (`TravelClass.ECONOMY == "Economy"`).

In [ ]:
class TravelClass(str, Enum):
    ECONOMY = "Economy"
    BUSINESS = "Business"


class BoardingPass(BaseModel):
    passenger_name: str = Field(description="As: Firstname Middlename Lastname")
    origin: str = Field(description="City of origin.")
    destination: str = Field(description="Destination city.")
    departure_date: date = Field(description="ISO-8601 formatted date; YYYY-MM-DD.")
    seat_row: int
    seat_letter: str
    cabin_bag_allowed: bool
    travel_class: TravelClass


response = client.responses.parse(
    model=MODEL,
    input=image_input("Extract the boarding pass fields into the provided schema."),
    text_format=BoardingPass,
)
boarding_pass = response.output_parsed
boarding_pass

The JSON schema Pydantic generates matches the one we wrote by hand quite well. There are a few differences though:

- In addition to `type` the Pydantic schema has a `title` which is just a capitalized rewriting of the field.
- The enum is defined in a separte `$defs` field and then referenced with `$ref`

In [ ]:
pydantic_schema = BoardingPass.model_json_schema()
for field in ["departure_date", "seat_row", "cabin_bag_allowed", "travel_class"]:
    print(f"{field}: hand-written {schema['properties'][field]}")
    print(f"{' ' * len(field)}  Pydantic     {pydantic_schema['properties'][field]}")
print("$defs:", pydantic_schema["$defs"])

Every field arrives as a Python value of the right type, so we can compute with it directly, without parsing strings:

In [ ]:
print("Departs on a", boarding_pass.departure_date.strftime("%A"))  # a datetime.date, not a string
print("Window seat:", boarding_pass.seat_letter in {"A", "F"})
print("Rows from the front:", boarding_pass.seat_row - 1)  # an int
if boarding_pass.travel_class is TravelClass.ECONOMY and boarding_pass.cabin_bag_allowed:
    print("Economy with a cabin bag")

### Failure modes

A schema makes the output *parseable*; it does not make it *correct*. The typical failures:

- **Hallucinated values**: a field the image doesn't show gets filled with a plausible guess.
- **Confused values**: the pass has several numbers (`23A`, `A12`, `042`, `2805`); the model can put the gate or the sequence number in `seat_row`.
- **Illegible input**: without the instruction "use null rather than guessing" (and a nullable field to allow it), the model has to invent a value. You will see this on an invoice in Exercise 3.
- **Schema violations** when the format is not enforced (sections 2–3): missing keys, other names, `"23A"` as the seat row, `"14NOV26"` as the date.

Pydantic catches the last kind at runtime (`BoardingPass.model_validate_json(...)` raises a `ValidationError`); the first three need evaluation against ground truth (M05).

In [ ]:
# Pydantic rejects data that doesn't match the type: each of these fails validation
for bad in [{"seat_row": "23A"}, {"departure_date": "14NOV26"}, {"travel_class": "First"}, {"cabin_bag_allowed": "1 x 8 kg"}]:
    try:
        BoardingPass.model_validate({**boarding_pass.model_dump(), **bad})
    except ValidationError as e:
        print(type(e).__name__, e, "\n")

## 6. Cost

Every call is billed by token: the input (prompt **and image**) and the output (including the reasoning tokens the model doesn't show). `usage_cost` from `agentic_workshop` reads the `usage` of a response and applies the prices in `agentic_workshop/costs.py`.

In [ ]:
print_cost(response.usage)

c = usage_cost(response.usage)
input_cost = (c["input_tokens"] - c["cached_tokens"]) * PRICES[MODEL]["input"] / 1e6
print(f"Input share of the cost: {input_cost / c['cost_usd']:.0%}")

The image is a large part of the input. Sending a long document (a 50-page policy PDF) on every call would be far more expensive: in M03 we show how to send only the relevant passages (RAG).

## Takeaways

- Asking for JSON in the prompt is not enough; JSON mode guarantees valid JSON, not the right keys.
- A strict JSON schema (or a Pydantic model, which generates one) guarantees the shape: the keys, the types (`string`, `integer`, `boolean`, a `date` string) and the allowed values of an `enum`.
- Nothing guarantees the values: allow `null` for what may be missing, and evaluate against ground truth.

## Exercises

Now apply the same steps to the medical invoices in `data/invoices/`: images of Groupe Mutuel claim forms. The `MedicalInvoice` model of Exercise 2 is the schema used for the rest of the day (it also lives in `agentic_workshop/invoices.py`, without the `tags` field).

In [ ]:
invoice_path = INVOICES_DIR / "INV-2025-002.png"
display(Image(filename=invoice_path, width=600))

### Exercise 1: A JSON schema for the invoice

Write a strict JSON schema for the invoice with these fields: `invoice_id`, `patient_name`, `patient_date_of_birth`, `insurance_product`, `policy_start_date`, `provider`, `provider_type`, `treatment_date`, `treatment_description`, `amount_chf` and `tags`, and extract INV-2025-002 with it.

`tags` is not an explicit field in the invoice, but would be a field useful for downstream search and processing, so it would be something reasonable to add in an actual working scenario.

- The dates (`patient_date_of_birth`, `policy_start_date`, `treatment_date`) are ISO-8601 date strings.
- `amount_chf` may be illegible: allow `null`.
- `tags` is a list of tags from a fixed list (an array of enum values). Several tags can apply to one invoice. The tags describe the type of care and its circumstances; all of them can be read off the invoice itself:
  - Type of care: `dental`, `alternative_medicine`, `medication`, `convalescence`, `consultation`, `surgery`
  - Circumstances: `cosmetic`, `accident`, `emergency`, `prescribed`

In [ ]:
class InvoiceTag(str, Enum):
    # Type of care
    DENTAL = "dental"
    ALTERNATIVE_MEDICINE = "alternative_medicine"
    MEDICATION = "medication"
    CONVALESCENCE = "convalescence"
    CONSULTATION = "consultation"
    SURGERY = "surgery"
    # Circumstances
    COSMETIC = "cosmetic"
    ACCIDENT = "accident"
    EMERGENCY = "emergency"
    PRESCRIBED = "prescribed"


TAGS = [tag.value for tag in InvoiceTag]    # Turn the enum into a list of strings for the schema

In [ ]:
# TODO (Exercise 1): write the invoice JSON schema, with an enum for the tags
invoice_schema = {
    "type": "object",
    "properties": {
        # TODO: Fill this in!
    }
}

In [ ]:
# SOLUTION
invoice_schema = {
    "type": "object",
    "properties": {
        "invoice_id": {"type": "string"},
        "patient_name": {"type": "string"},
        "patient_date_of_birth": {"type": "string", "format": "date", "description": "ISO-8601 formatted date; YYYY-MM-DD."},
        "insurance_product": {"type": "string"},
        "policy_start_date": {"type": "string", "format": "date", "description": "ISO-8601 formatted date; YYYY-MM-DD."},
        "provider_name": {"type": "string"},
        "provider_type": {"type": "string"},
        "treatment_date": {"type": "string", "format": "date", "description": "ISO-8601 formatted date; YYYY-MM-DD."},
        "treatment_description": {"type": "string"},
        "notes": {"type": "string"},
        "amount_chf": {"type": ["number", "null"], "description": "Use null if any digit is not clearly legible."},
        "tags": {
            "type": "array",
            "items": {"type": "string", "enum": TAGS},
            "description": "Every tag that applies to this invoice.",
        },
    },
    "required": ["invoice_id", "patient_name", "patient_date_of_birth", "insurance_product", "policy_start_date", "provider_name", "provider_type", "treatment_date", "treatment_description", "notes", "amount_chf", "tags"],
    "additionalProperties": False,
}

In [ ]:
# See that it works!
response = client.responses.create(
    model=MODEL,
    input=image_input("Extract the invoice fields.", invoice_path),
    text={"format": {"type": "json_schema", "name": "MedicalInvoice", "schema": invoice_schema, "strict": True}},
)
json.loads(response.output_text)

### Exercise 2: The invoice as a Pydantic model

Write `MedicalInvoice` as a Pydantic model with all the fields of the claim form, plus `tags` as a list of `InvoiceTag`. Extract INV-2025-002 with `client.responses.parse`.

In [ ]:
# TODO (Exercise 2): write MedicalInvoice as a Pydantic model
class MedicalInvoice(BaseModel):
    # TODO: Fill this in!
    pass

In [ ]:
# SOLUTION
class MedicalInvoice(BaseModel):
    invoice_id: str
    patient_name: str
    patient_date_of_birth: str = Field(description="As YYYY-MM-DD.")
    policy_holder_number: str
    insurance_product: str
    policy_start_date: str = Field(description="As YYYY-MM-DD.")
    provider_name: str
    provider_type: str
    treatment_date: str = Field(description="As YYYY-MM-DD.")
    treatment_description: str
    notes: str | None = None
    amount_chf: float | None = Field(description="Use null if any digit is not clearly legible.")
    tags: list[InvoiceTag] = Field(description="Every tag that applies to this invoice.")


In [ ]:
# And run to test it!
response = client.responses.parse(
    model=MODEL,
    input=image_input("Extract the invoice fields into the provided schema.", invoice_path),
    text_format=MedicalInvoice,
)
invoice = response.output_parsed
invoice

### Exercise 3: Illegible values and wrong tags

1. INV-2025-012's amount is smudged. Extract it with the `MedicalInvoice` Pydantic model.
2. INV-2025-010 says "not an accident" and INV-2025-011 "not an emergency". Does the model add those tags anyway? The enum guarantees a valid tag, not the right one. Try it out with `OLD_MODEL` as well, does the result change?

#### Exercise 3.1

Run extraction against INV-2025-012 which has the amount field smudged. See below.

In [ ]:
display(Image(filename=INVOICES_DIR / "INV-2025-012.png", width=600))

In [ ]:
# TODO extract the smudged invoice, with a null value rather than guessing
response = ...
print(f"amount chf: {response.output_parsed.amount_chf}")

In [ ]:
# SOLUTION
response = client.responses.parse(
    model=MODEL,
    input=image_input("Extract the invoice fields.", path=INVOICES_DIR / "INV-2025-012.png"),
    text_format=MedicalInvoice,
)
print("amount_chf:", response.output_parsed.amount_chf)

In [ ]:
# SOLUTION (JSON schema)
response = client.responses.create(
    model=MODEL,
    input=image_input(prompt="Extract the invoice fields.", path=INVOICES_DIR / "INV-2025-012.png"),
    text={"format": {"type": "json_schema", "name": "MedicalInvoice", "schema": invoice_schema, "strict": True}},
)
print("amount_chf:", json.loads(response.output_text)["amount_chf"])

#### Exercise 3.2

Run extraction against INV-2025-010 and INV-2025-011 to see if we can trick the model into outputting tags it should not add.

Test this with both `MODEL` and `OLD_MODEL` to see if we get any differences.

In [ ]:
invoices = ["INV-2025-010", "INV-2025-011"]
for invoice in invoices:
    display(Image(filename=INVOICES_DIR / f"{invoice}.png", width=600))

In [ ]:
# TODO check the tags of the two trap invoices
for invoice in invoices:
    response = ...
    print(response.output_parsed.notes, "->", [tag.value for tag in response.output_parsed.tags])

In [ ]:
# SOLUTION
for invoice in invoices:
    response = client.responses.parse(
        model=MODEL,
        input=image_input("Extract the invoice fields.", INVOICES_DIR / f"{invoice}.png"),
        text_format=MedicalInvoice,
    )
    parsed = response.output_parsed
    print(invoice, parsed.notes, "->", [tag.value for tag in parsed.tags])

In [ ]:
# SOLUTION (JSON schema)
for invoice in invoices:
    response = client.responses.create(
        model=MODEL,
        input=image_input(prompt="Extract the invoice fields.", path=INVOICES_DIR / f"{invoice}.png"),
        text={"format": {"type": "json_schema", "name": "MedicalInvoice", "schema": invoice_schema, "strict": True}},
    )
    extracted = json.loads(response.output_text)
    print(invoice, extracted["notes"], "->", extracted["tags"])

#### Exercise 3.3 (Optional Extra) - Only if you have time

Go back to Exercises 3.1 and 3.2 and re-do them with the JSON schema extraction method. Do you prefer working with the ouput of Pydantic or the plain JSON schema?

### 4: Comparing Extraction Methods

This part is already set up and relies on your previous solutions. So make sure you have completed exercises 1 and 2 before running this. It does the following:

1. Extracts three of the trickier invoices (INV-2025-012, INV-2025-006 and INV-2025-002) with both methods: the JSON schema (Exercise 1) and Pydantic (Exercise 2).
2. Runs each method 1–2 times. Do the results vary from run to run?
3. Compares both against the ground truth in `data/invoices/ground_truth/` (`load_ground_truth` from `agentic_workshop`), and prints every field that deviates from it.
4. Computes the total cost of all the calls.

Play around with it a little and see if there are any differences when you use either `MODEL` or `OLD_MODEL` to run the test.

In [ ]:
# This is here to stop you running this section uninitentionally with a "run all" command as it will take rougly 1 minute to run and will call the LLM several times.
raise RuntimeError("Manually run the next cell!")

In [ ]:
from agentic_workshop.invoices import load_ground_truth

TEST_MODEL = OLD_MODEL

FIELDS = ["invoice_id", "patient_name", "insurance_product", "treatment_date", "treatment_description", "amount_chf"]

usages = []  # the usage of every call, for the total cost

def extract_with_schema(path):
    response = client.responses.create(
        model=TEST_MODEL,
        input=image_input("Extract the invoice fields.", path),
        text={"format": {"type": "json_schema", "name": "MedicalInvoice", "schema": invoice_schema, "strict": True}},
    )
    usages.append(response.usage)
    return json.loads(response.output_text)

def extract_with_pydantic(path):
    response = client.responses.parse(
        model=TEST_MODEL,
        input=image_input("Extract the invoice fields.", path),
        text_format=MedicalInvoice,
    )
    usages.append(response.usage)
    return response.output_parsed.model_dump()

# Trickier invoices: a smudged amount (012), a long description (006), a large amount on a multi-tooth treatment (002)
reduced_set = [INVOICES_DIR / f"INV-2025-{n}.png" for n in ["012", "006", "002"]]

for run in range(2):
    correct = {"schema": 0, "pydantic": 0}
    deviations = []  # every field that differs from the ground truth in this run
    for path in reduced_set:
        truth = load_ground_truth(path.stem)["extraction"]
        extracted = {"schema": extract_with_schema(path), "pydantic": extract_with_pydantic(path)}
        for method, values in extracted.items():
            for f in FIELDS:
                if values.get(f) == truth[f]:
                    correct[method] += 1
                else:
                    deviations.append(f"  {path.stem} {method:<8} {f}: got {values.get(f)!r}, expected {truth[f]!r}")
    total = len(reduced_set) * len(FIELDS)
    print(f"run {run}: fields correct, schema {correct['schema']}/{total}, pydantic {correct['pydantic']}/{total}")
    print("\n".join(deviations) if deviations else "  no deviations")

print(f"\nTotal over {len(usages)} calls:")
print_cost(usages, model=TEST_MODEL)